# PTCG self-play PPO — Kaggle training

Upload this notebook to Kaggle and press **Run All** (or **Save Version → Save & Run All** to train in the background for up to 12 h).

Before running, in the notebook's right-hand panel:

- **Accelerator:** `GPU T4 x2`. The two GPUs split the work: GPU 0 runs the PPO update in fp16 while GPU 1 plays the next rollout's games (`PIPELINE`, `USE_AMP`). With one GPU both run on it; the P100 is not supported by recent PyTorch builds.
- **Internet:** on (needed for `git clone` and `pip install`)
- **Weights & Biases (optional):** *Add-ons → Secrets*, add a secret named `WANDB_API_KEY` with your key from https://wandb.ai/authorize and tick it for this notebook. Without it the run logs to W&B offline (files under `runs/kaggle/wandb/`, upload later with `wandb sync`).

Everything worth keeping lands in `/kaggle/working/` and shows up in the notebook's **Output**:

| | |
| --- | --- |
| `runs/kaggle/final.zip` | the trained MaskablePPO model (resume from it) |
| `runs/kaggle/latest.zip` | rewritten at every snapshot, survives a killed session |
| `runs/kaggle/checkpoints/` | one every `SAVE_EVERY` steps |
| `runs/kaggle/tb/` | TensorBoard logs (the same values go to W&B, including `vs_deck/<deck>` win rates) |
| `submissions/kaggle_final.zip` | competition submission built and checked by `ptcg_rl.export` |

What training uses (all set in the next cell):

- the agent's deck `decks/main_agent.csv` against `decks/training_pool/*.csv`, drawn by `decks/training_pool/weights.json` (Kaggle meta share, more weight on the matchups the 216M submission lost);
- the decision rules the submission also plays by: damage counters "in any way you like" go where they take the most Prize cards, and a turn that offers a drawback-free attack is not ended without attacking (`USE_RULES`).

**Resuming:** add a previous version's output as an input dataset, then set `RESUME` below to its `latest.zip` or `final.zip` (or leave `RESUME = "auto"` to pick up the newest one found under `/kaggle/input`).

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/nikephorosphokas99/ptcg-ai-sim.git"
BRANCH = "self-play-ppo"

# Kaggle stops a session at 12 h. Training stops itself after MAX_HOURS of training time,
# leaving time for setup, export and evaluation.
MAX_HOURS = 10.5
TIMESTEPS = 500_000_000      # upper bound; MAX_HOURS normally ends the run first
ARCH = "entity"             # entity (v3) | pointer | flat
# Opponent decks, relative to rl-train/. Without them every game is a mirror match:
# the default (decks/opponents/) does not exist in the repo.
OPPONENT_DECKS = "decks/training_pool/*.csv"
OPPONENT_WEIGHTS = "decks/training_pool/weights.json"   # None draws every deck equally often
SELF_PLAY_FRAC = 0.5         # share of games against the current learner (the rest: snapshots, random)
USE_RULES = True             # counter planner + no passing on a free attack, as the submission plays
PIPELINE = True              # update on one GPU while the next rollout is collected (on the other GPU if there are two)
USE_AMP = True               # fp16 PPO update on the T4's tensor cores
BOT_FRAC = 0.0               # share of games vs the rule-based bots in benchmarks/
SAVE_EVERY = 5_000_000
SNAPSHOT_EVERY = 1_000_000
RESUME = "auto"              # "auto", None, or a path to a model .zip
EXTRA_ARGS = []              # any other ptcg_rl.train flags, e.g. ["--lr-end", "3e-5"]

# Weights & Biases: the API key comes from the Kaggle secret WANDB_API_KEY (see above).
WANDB_PROJECT = "ptcg-ai-sim"    # None turns W&B off
WANDB_ENTITY = None              # your W&B user or team; None uses the key's default
WANDB_RUN_NAME = None            # None: "kaggle-<date>-<time>"
WANDB_LOG_MODEL = True           # upload checkpoints and final.zip as W&B model artifacts

EVAL_GAMES = 200             # vs the random player after training (0 to skip)
BENCHMARK_GAMES = 20         # per rule-based opponent after training (0 to skip)

WORK = "/kaggle/working"
REPO_DIR = "/kaggle/temp/ptcg-ai-sim"   # code lives outside /kaggle/working so it isn't saved as output
RUN_DIR = f"{WORK}/runs/kaggle"
SUB_DIR = f"{WORK}/submissions"

In [ ]:
import os, subprocess, sys, shutil, glob, time

def run(cmd, cwd=None, check=True):
    """Run a command, streaming its output into the notebook log line by line."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.send_signal(2)  # SIGINT: train.py saves final.zip on the way out
        proc.wait()
        raise
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc

run(["nvidia-smi"], check=False)
print("CPUs:", os.cpu_count())

In [ ]:
# ---- Dependencies ----
# Every package the training, export, evaluation and benchmark code imports beyond the
# standard library (rl-train/requirements.txt plus wandb), listed here so the install doesn't
# depend on the clone. Kaggle's image already has torch, numpy and tensorboard; pip leaves a
# satisfying version alone, so its CUDA build of torch is kept. git is preinstalled.
DEPENDENCIES = [
    "torch>=2.2",
    "numpy",
    "gymnasium>=1.0",
    "stable-baselines3>=2.3",
    "sb3-contrib>=2.3",
    "tensorboard",
    "wandb",
]
run([sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES])
run([sys.executable, "-c", "import torch, numpy, gymnasium, stable_baselines3, sb3_contrib, tensorboard, wandb; "
     "print('torch', torch.__version__, '| numpy', numpy.__version__, '| gymnasium', gymnasium.__version__, "
     "'| sb3', stable_baselines3.__version__, '| sb3-contrib', sb3_contrib.__version__, "
     "'| tensorboard', tensorboard.__version__, '| wandb', wandb.__version__)"])

In [ ]:
# ---- Get the code ----
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"])
TRAIN_DIR = f"{REPO_DIR}/rl-train"
os.chmod(f"{TRAIN_DIR}/engine/sample_submission/cg/libcg.so", 0o755)

# The branch must have the code this notebook drives; an unpushed branch would otherwise
# train the old code without saying so.
REQUIRED = ["ptcg_rl/ppo.py", "decks/training_pool/weights.json", "decks/training_pool/festival_thwackey.csv"]
missing = [f for f in REQUIRED if not os.path.exists(f"{TRAIN_DIR}/{f}")]
train_src = open(f"{TRAIN_DIR}/ptcg_rl/train.py").read()
missing += [flag for flag in ("--pipeline", "--amp", "--collect-device", "--opponent-weights", "--wandb-project")
            if flag not in train_src]
if missing:
    raise RuntimeError(f"{BRANCH} on GitHub is missing {missing}: push the latest commits "
                       f"(git push origin {BRANCH}) and run again.")
print("code check passed")

In [ ]:
# ---- Devices ----
# Every GPU torch can see gets a real op: one it has no kernels for (e.g. a P100 on a
# recent build) only fails on first use.
import torch
GPUS = []
for i in range(torch.cuda.device_count()):
    try:
        (torch.randn(256, 256, device=f"cuda:{i}") @ torch.randn(256, 256, device=f"cuda:{i}")).sum().item()
        props = torch.cuda.get_device_properties(i)
        print(f"cuda:{i}: {props.name}, {props.total_memory / 2**30:.1f} GB, sm_{props.major}{props.minor}")
        GPUS.append(i)
    except Exception as e:
        print(f"cuda:{i} unusable: {e}")
if not GPUS:
    print("WARNING: no usable GPU; training on CPU will be very slow. Set Accelerator to GPU T4 x2.")

# Update on the first GPU; games, the sample store and the snapshot opponents on the second.
DEVICE = f"cuda:{GPUS[0]}" if GPUS else "cpu"
COLLECT_DEVICE = f"cuda:{GPUS[1]}" if len(GPUS) > 1 else DEVICE
DEVICE_ARGS = ["--device", DEVICE, "--collect-device", COLLECT_DEVICE]
if PIPELINE and GPUS:
    DEVICE_ARGS += ["--pipeline"]
if USE_AMP and GPUS:
    DEVICE_ARGS += ["--amp"]

# Kaggle gives 4 CPUs: keep one for the main process (game inference + the update thread).
WORKERS = max(1, (os.cpu_count() or 2) - 1)
MATCHES_PER_WORKER = 64

# The sample store lives on COLLECT_DEVICE (README: ~11 KB per sample for the v3 policy,
# 6 x 65,536 by default = 4.3 GB); smaller GPUs get a smaller store and minibatch.
STORE_ARGS = []
if GPUS and torch.cuda.get_device_properties(GPUS[-1]).total_memory < 12 * 2**30:
    STORE_ARGS = ["--store", "196608", "--batch-size", "1024"]
print("devices:", " ".join(DEVICE_ARGS), "| workers:", WORKERS, "| store args:", STORE_ARGS)

In [ ]:
# ---- Weights & Biases ----
# The key is read from the Kaggle secret WANDB_API_KEY and handed to the training process
# through its environment; it is never printed or written to the outputs.
WANDB_ARGS = []
if WANDB_PROJECT:
    key = os.environ.get("WANDB_API_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception as e:
            print(f"No WANDB_API_KEY secret ({type(e).__name__}); logging to W&B offline under {RUN_DIR}/wandb")
    if key:
        os.environ["WANDB_API_KEY"] = key
    run_name = WANDB_RUN_NAME or time.strftime("kaggle-%Y%m%d-%H%M")
    WANDB_ARGS = ["--wandb-project", WANDB_PROJECT, "--wandb-name", run_name,
                  "--wandb-mode", "online" if key else "offline"]
    if WANDB_ENTITY:
        WANDB_ARGS += ["--wandb-entity", WANDB_ENTITY]
    if WANDB_LOG_MODEL:
        WANDB_ARGS += ["--wandb-log-model"]
print("W&B:", " ".join(WANDB_ARGS) or "off")

In [ ]:
# ---- Smoke test: the engine loads and the decks are valid ----
OPP_DECK_FILES = sorted(glob.glob(f"{TRAIN_DIR}/{OPPONENT_DECKS}"))
assert OPP_DECK_FILES, f"no opponent decks match {OPPONENT_DECKS}"
run([sys.executable, "-c", """
import sys
from ptcg_rl.env import Match
from ptcg_rl.engine import AGENT_DECK
m = Match(agent_deck=str(AGENT_DECK), opponent_decks=sys.argv[1:])
m.close()
print('engine OK; opponent decks:', len(sys.argv) - 1)
""", *OPP_DECK_FILES], cwd=TRAIN_DIR)

In [ ]:
# ---- Resume from a previous run, if any ----
resume = RESUME
if resume == "auto":
    found = [p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True)
             if os.path.basename(p) in ("latest.zip", "final.zip") or "/checkpoints/" in p]
    resume = max(found, key=os.path.getmtime) if found else None
print("resume from:", resume or "nothing (new model)")

In [ ]:
# ---- Pre-flight: a two-minute run with the real flags ----
# One small update and an export, with the same devices, pipeline, fp16, rules and decks as
# the real run, so a problem shows up now rather than hours into the GPU quota. W&B is off here.
PREFLIGHT_DIR = "/kaggle/temp/preflight"
shutil.rmtree(PREFLIGHT_DIR, ignore_errors=True)
run([sys.executable, "-u", "-m", "ptcg_rl.train", "--run", PREFLIGHT_DIR, "--arch", ARCH,
     "--timesteps", 8192, "--rollout", 4096, "--batch-size", 1024, "--store", 32768,
     "--workers", WORKERS, "--matches-per-worker", 8, "--snapshot-every", 4096, "--save-every", 8192,
     *DEVICE_ARGS, "--opponent-decks", *OPP_DECK_FILES,
     *(["--opponent-weights", f"{TRAIN_DIR}/{OPPONENT_WEIGHTS}"] if OPPONENT_WEIGHTS else []),
     *([] if USE_RULES else ["--no-rules"])], cwd=TRAIN_DIR)
run([sys.executable, "-m", "ptcg_rl.export", f"{PREFLIGHT_DIR}/final.zip", "--out", PREFLIGHT_DIR,
     "--name", "preflight", "--check-games", 4, *([] if USE_RULES else ["--no-rules"])], cwd=TRAIN_DIR)
print("pre-flight passed")

In [ ]:
# ---- Train ----
os.makedirs(RUN_DIR, exist_ok=True)
cmd = [sys.executable, "-u", "-m", "ptcg_rl.train",
       "--run", RUN_DIR,
       "--arch", ARCH,
       "--timesteps", TIMESTEPS,
       "--max-hours", MAX_HOURS,
       "--workers", WORKERS,
       "--matches-per-worker", MATCHES_PER_WORKER,
       "--save-every", SAVE_EVERY,
       "--snapshot-every", SNAPSHOT_EVERY,
       "--self-play-frac", SELF_PLAY_FRAC,
       "--bot-frac", BOT_FRAC,
       *DEVICE_ARGS,
       "--opponent-decks", *OPP_DECK_FILES,
       *STORE_ARGS, *WANDB_ARGS, *EXTRA_ARGS]
if OPPONENT_WEIGHTS:
    cmd += ["--opponent-weights", f"{TRAIN_DIR}/{OPPONENT_WEIGHTS}"]
if not USE_RULES:
    cmd += ["--no-rules"]
if resume:
    cmd += ["--resume", resume]
t0 = time.time()
run(cmd, cwd=TRAIN_DIR)
print(f"training finished in {(time.time() - t0) / 3600:.2f} h")
run(["ls", "-la", RUN_DIR, f"{RUN_DIR}/checkpoints"], check=False)

In [ ]:
# ---- Export a competition submission ----
# Checks the bundle's network against the trained policy and plays full games with it
# loaded the way the competition harness loads it, then writes submissions/kaggle_final.zip.
# The bundle carries rules.json, so it plays the rule-decided picks exactly as training did.
MODEL = f"{RUN_DIR}/final.zip"
run([sys.executable, "-m", "ptcg_rl.export", MODEL, "--out", SUB_DIR, "--name", "kaggle_final",
     *([] if USE_RULES else ["--no-rules"])], cwd=TRAIN_DIR)
SUBMISSION = f"{SUB_DIR}/kaggle_final.zip"

In [ ]:
# ---- Evaluate ----
if EVAL_GAMES:
    run([sys.executable, "-m", "ptcg_rl.evaluate", MODEL, "--games", EVAL_GAMES,
         "--workers", WORKERS, "--device", DEVICE, "--opponent-decks", *OPP_DECK_FILES],
        cwd=TRAIN_DIR, check=False)
if BENCHMARK_GAMES:
    run([sys.executable, "-m", "ptcg_rl.benchmark", SUBMISSION, "--games", BENCHMARK_GAMES,
         "--parallel", WORKERS], cwd=TRAIN_DIR, check=False)

In [ ]:
# ---- Outputs ----
run(["du", "-sh", RUN_DIR, SUB_DIR], check=False)
print("\nDownload from the notebook's Output tab:")
for p in sorted(glob.glob(f"{WORK}/**/*.zip", recursive=True)):
    print(f"  {p}  ({os.path.getsize(p) / 2**20:.1f} MB)")